# Session 1 — Instructor Solutions
**Python for Mechanical Engineers | Dr. Nuha Aljuneidi**

Keep this notebook private until students complete the design challenge.

In [ ]:
import math
import numpy as np

## Checkpoint 1 — Modified sensible-energy case

In [ ]:
mass_kg = 25.0
cp_J_kgK = 4180.0
T_initial_C, T_final_C = 20.0, 75.0
energy_J = mass_kg * cp_J_kgK * (T_final_C - T_initial_C)
print(f"Energy required: {energy_J/1e6:.3f} MJ ({energy_J/3.6e6:.3f} kWh)")
# The energy increases because both mass and temperature rise increase.
# A temperature interval has the same numerical size in °C and K.

## Reusable sensible-energy function

In [ ]:
def sensible_energy_kWh(mass_kg, cp_J_kgK, T_initial_C, T_final_C):
    """Return signed sensible energy in kWh. Positive means heating."""
    if mass_kg <= 0 or cp_J_kgK <= 0:
        raise ValueError("Mass and heat capacity must be positive.")
    energy_J = mass_kg * cp_J_kgK * (T_final_C - T_initial_C)
    return energy_J / 3.6e6

assert abs(sensible_energy_kWh(12, 4180, 20, 60) - 0.5573333) < 1e-6
print(f"{sensible_energy_kWh(12, 4180, 20, 60):.3f} kWh")

## Checkpoint 2 — Reynolds number and regime

In [ ]:
def pipe_flow_regime(rho_kg_m3, velocity_m_s, diameter_m, mu_Pa_s):
    values = (rho_kg_m3, velocity_m_s, diameter_m, mu_Pa_s)
    if any(value <= 0 for value in values):
        raise ValueError("All inputs must be positive.")
    Re = rho_kg_m3 * velocity_m_s * diameter_m / mu_Pa_s
    regime = "laminar" if Re < 2300 else "transitional" if Re < 4000 else "turbulent"
    return Re, regime

for velocity in [0.05, 0.10, 0.50, 1.00]:
    Re, regime = pipe_flow_regime(997, velocity, 0.025, 0.00089)
    print(f"V={velocity:.2f} m/s: Re={Re:,.0f}, {regime}")

## Checkpoint 3 — Wall heat loss and insulation improvement

In [ ]:
def wall_heat_loss_W(area_m2, Ti_C, To_C, h_i, h_o, thickness_m, k_W_mK):
    if min(area_m2, h_i, h_o, thickness_m, k_W_mK) <= 0:
        raise ValueError("Area, coefficients, thickness, and conductivity must be positive.")
    R_total = 1/h_i + thickness_m/k_W_mK + 1/h_o
    U = 1/R_total
    return U * area_m2 * (Ti_C - To_C)

q_original = wall_heat_loss_W(20, 22, -5, 10, 25, 0.08, 0.04)
q_improved = wall_heat_loss_W(20, 22, -5, 10, 25, 0.16, 0.04)
reduction_pct = 100 * (q_original - q_improved) / q_original
print(f"Original: {q_original:.1f} W ({q_original/1000:.3f} kW)")
print(f"Double insulation: {q_improved:.1f} W; reduction = {reduction_pct:.1f}%")

## Design challenge — Storage-tank heating time

In [ ]:
def tank_heating_time(mass_kg, T_initial_C, T_final_C, power_kW, efficiency, cp_J_kgK=4180):
    if mass_kg <= 0 or power_kW <= 0 or cp_J_kgK <= 0:
        raise ValueError("Mass, power, and heat capacity must be positive.")
    if not 0 < efficiency <= 1:
        raise ValueError("Efficiency must be greater than 0 and no more than 1.")
    if T_final_C <= T_initial_C:
        raise ValueError("Final temperature must exceed initial temperature.")
    energy_J = mass_kg * cp_J_kgK * (T_final_C - T_initial_C)
    time_s = energy_J / (efficiency * power_kW * 1000)
    return time_s/60, time_s/3600

minutes, hours = tank_heating_time(150, 18, 60, 4.5, 0.88)
print(f"Heating time: {minutes:.1f} min ({hours:.2f} h)")
for power in range(2, 9):
    _, hours = tank_heating_time(150, 18, 60, power, 0.88)
    print(f"{power} kW -> {hours:.2f} h")

## Advanced extension — Tank heat loss
For a constant ambient temperature, $mc_p\,dT/dt=\eta P-UA(T-T_a)$. The steady limiting temperature is $T_\infty=T_a+\eta P/(UA)$ and
$T(t)=T_\infty+(T_0-T_\infty)e^{-UA t/(mc_p)}$.
A target at or above $T_\infty$ is unreachable in this idealized model.

In [ ]:
def tank_heating_time_with_loss(mass_kg, T0_C, target_C, power_kW, efficiency, UA_W_K, ambient_C, cp_J_kgK=4180):
    if min(mass_kg, power_kW, efficiency, UA_W_K, cp_J_kgK) <= 0 or efficiency > 1:
        raise ValueError("Check the positive inputs and efficiency.")
    T_infinity = ambient_C + efficiency * power_kW * 1000 / UA_W_K
    if target_C >= T_infinity:
        raise ValueError(f"Target is unreachable; limiting temperature is {T_infinity:.1f}°C.")
    ratio = (target_C - T_infinity) / (T0_C - T_infinity)
    time_s = -mass_kg * cp_J_kgK / UA_W_K * math.log(ratio)
    return time_s/3600, T_infinity

hours, limit_C = tank_heating_time_with_loss(150, 18, 60, 4.5, 0.88, 12, 20)
print(f"With heat loss: {hours:.2f} h; limiting temperature: {limit_C:.1f}°C")